# Simple forecasts on the validation years

Scores the three simple forecasts (persistence, climatology, damped anomaly) on 2020-2022.
Every model has to beat the best of these at each lead time. All three are fitted on the training years (2000-2019) only.

In [ ]:
import sys
sys.path.insert(0, "../data_scripts")  # relative to the notebook's folder

import matplotlib.pyplot as plt
import pandas as pd
from prepare_data import load_data
from scoring import simple_forecasts, score

In [ ]:
data = load_data()
rows = data[data.usable & (data.period == "validation")]
print(f"{len(rows):,} validation rows at {rows.STATION.nunique()} stations")

In [ ]:
forecasts = simple_forecasts(data, rows)
scores = {name: score(rows, pred) for name, pred in forecasts.items()}
mae = pd.DataFrame({name: s.MAE for name, s in scores.items()})
rmse = pd.DataFrame({name: s.RMSE for name, s in scores.items()})
mae.round(2)

In [ ]:
rmse.round(2)

In [ ]:
# Categorical slots 1-3 of the validated default palette; markers differ so identity is not color alone.
styles = {"persistence": ("#2a78d6", "o"), "climatology": ("#eb6834", "s"), "damped anomaly": ("#1baf7a", "D")}

fig, ax = plt.subplots(figsize=(7, 4))
for name, (color, marker) in styles.items():
    ax.plot(mae.index, mae[name], color=color, marker=marker, markersize=6, linewidth=2, label=name)
ax.set(xlabel="Lead time (days)", ylabel="MAE (°C)", title="Simple forecast error on 2020-2022", xticks=list(mae.index))
ax.grid(axis="y", color="#e5e4e0")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.show()

## Takeaways

* Damped anomaly has the lowest MAE and RMSE at every lead time, so it is the forecast to beat.
* Persistence beats climatology only at lead 1.
* From lead 4 on, damped anomaly is within about 0.06 °C of climatology: today's weather adds almost nothing past 3 days.